# Modelo Simplificado de Ótimo Económico FV + Bateria

O **objetivo** deste modelo é encontrar o par $(R_{PV}, R_{BAT})$ que minimiza o custo anual de energia sem recorrer a simulação horária.

## 1. Princípio Fundamental

O sistema é modelado através de fluxos energéticos médios diários. A energia consumida pode vir de:

* Produção FV instantânea
* Energia FV armazenada na bateria
* Rede

O ótimo económico ocorre quando: _o benefício marginal de aumentar FV ou bateria é igual ao respetivo custo marginal anualizado._

## 2. Variável de referência

Energia diária de referência $E_{ref}$

### 2.1 Média das médias diárias mensais

$E_{ref} = \frac{1}{12}\sum_{m=1}^{12} E_{dia,m}$

$E_{dia,m} = \frac{E_{mês}}{nr\space dias_{mês}}$

### 2.2 Média anual

$E_{ref} = \frac{E_{anual}}{365}$

## 3. Indicadores referência

### 3.1 Índice FV

$R_{PV} = \frac{E_{PV,dia}}{E_{ref}}$

Interpretação:
* 1.0 -> produção média diária do PV igual ao consumo médio diário.
* 1.5 -> produz 150% do consumo médio diário

### 3.2 Índice Bateria

$R_{BAT} = \frac{C_{bat}}{E_{ref}}$

Interpretação:
* 0.5 -> armazena 50% do consumo médio diário
* 1.0 -> armazena 1 dia completo

## 4. Produção PV

$E_{PV,dia} = R_{PV}*E_{ref}$ $(kWh/dia)$

## 5. Autoconsumo direto

Parâmetro: $a$

fracção correspondente à simulteineadade produção e consumo. Tipicamente $0.25 \leq a \leq 0.4$

Energia autoconsumida diretamente: $E_{dir} = a * E_{PV}$

## 6. Excedente PV

$E_{exc} = (1 - a)*E_{PV}$

## 7. Saturação da bateria

$f(R_{BAT}) = 1 - e^{-kR_{BAT}}$

onde $2 \leq k \leq 5$, parâmetro que mede a rapidez da saturação:
* k baixo: 
	* saturação lenta há beneficio em aumentar a capacidade da bateria.
	* Perfil tipico: muito FV, consumo noturno elevado
* k alto: 
	* saturação rápida, poucos kWh já capturam quase todo o benefício. Baterias grande raramente compensam.
	* Perfil tipico: consumo fora vazio baixo, poucos excedentes

<img src="./funcao_saturacao_bateria.png" width="500" />

## 8. Eficiência da bateria

Eficiência round-trip:  $\eta_{rt}$

tipicamente: $ 0.90 \leq \eta_{rt} \leq 0.95$

## 9. Energia recuparada pela bateria

$E_{bat} = \eta(rt) * E_{exc} * f(R_{BAT})$

## 10. Energia total servida pelo sistema

$E_{serv} = E_{dir} + E_{bat}$

Substituindo:

$E_{serv} = E_{PV}[a + \eta_{rt}(1-a)f(R_{BAT})]$

Onde:

$E_{serv} = R_{PV}E_{ref}[a + \eta_{rt}(1-a)f(R_{BAT})]$

## 11. Energia comprada à rede

$E_{grid} = max(0, E_{ref} - E_{serv})$

## 12. Custos anualizados

### 12.1 FV

$C_{PV} = CRF_{PV} [-]* CAPEX_{PV} [€/kWp]* P_{PV} [kWp]$

Onde:

$P_{PV} = \frac{R_{PV}E_{ref}365}{Y}$, potência instalada.

### 12.2 Bateria

$C_{BAT} = CRF_{BAT} [-] * CAPEX_{BAT} [€/kWh]* C_{bat} [kWh]$

Onde:

$C_{bat} = R_{BAT} E_{ref}$

### 12.3 Rede

$C_{grid} = p_{grid} * E_{grid} * 365$

## 13. Função objetivo

Minimizar:

$C_{total} = C_{PV} + C_{BAT} + C_{grid}$


In [7]:
import pandas as pd
import numpy as np

Parâmetros de entrada do modelo

* $E_{anual}$ : consumo anual [kWh]
* $a$ : indice de auto-consumo
* $\eta_{rt}$ : eficiência da bateria
* $Y$ : yield específico, de acordo com a localização do sistema [kWh/kWp]
* $CAPEX_{PV}$ : custo instalação FV por kWp [€/kWp]
* $CAPEX_{BAT}$ : custo instalação BAT por kWh [€/kWh]
* $p_{grid}$ : preço compra à rede [€/kWh]. Se bihorário usar preço fora vazio.


In [19]:
params = {
    "consumo_anual": 3000, # kWh
    "a": 0.30,
    "eff_bat": 0.9, 
    "Y": 1500,  # kWh/kWp
    "capex_pv": 1000, #€/kWp
    "capex_bat": 200, #€/kWh
    "p_grid": 0.13, #€/kWh
    "frac_consumo_fora_vazio": 0.68, # 
    "tempo_vida_pv": 20, # anos
    "tempo_vida_bat": 10, # anos
    "taxa_desconto": 0.05, #
}

In [22]:
def _saturacao_bateria(k, r_bat):
    """ Função de saturação da bateria.
    """
    return 1 - np.exp(-k * r_bat)

def _crf(r, n):
    """Calcula o CRF (Capital Recovery Factor) para uma taxa de desconto r e um número de anos n.
    
    Parameters
    ----------
    r : float
        Taxa de desconto (em decimal, por exemplo, 0.05 para 5%)
    n : int
        Número de anos do projeto

    Returns
    -------
    crf : float
        O CRF correspondente à taxa de desconto e ao número de anos fornecidos
    """
    return r * (1 + r)**n / ((1 + r)**n - 1)

def calcula_funcao_objectivo(r_pv, r_bat, k, params):
    """ Calcula o custo anual.
    """

    e_ref = params["consumo_anual"] / 365 # kWh/dia
    # e_pv_dia = r_pv * e_ref # kWh/dia

    # # energia consumida diretamente do PV
    # e_dir = params["a"] * e_pv_dia
    # e_exc = (1-params["a"]) * e_pv_dia
    # e_bat = params["eff_bat"] * _saturacao_bateria(k, r_bat) * e_exc
    e_serv = r_pv * e_ref * (params["a"] + params["eff_bat"]*(1 - params["a"])*_saturacao_bateria(k, r_bat)) # kWh/dia
    e_grid = max(0, e_ref - e_serv) # kWh/dia

    # custos pv
    pot_pv = r_pv * e_ref * 365 / params["Y"] # kWp
    custo_pv = pot_pv * params["capex_pv"] * _crf(params["taxa_desconto"], params["tempo_vida_pv"]) # €

    # custos bateria
    custo_bat = r_bat * e_ref * params["capex_bat"] * _crf(params["taxa_desconto"], params["tempo_vida_bat"]) # €

    # custos rede
    custo_grid = e_grid * 365 * params["p_grid"] # €

    return custo_pv + custo_bat + custo_grid

def encontra_indices_minimizam_custo(params):
    """ Encontra par (R_pv, R_bat) que minimiza o custo.
    """
    # Estimar parametro k 
    r_bat50 = params["frac_consumo_fora_vazio"] * 0.5
    k = np.log(2) / r_bat50

    # Procura por grid search
    r_pv_range = np.arange(0.0, 2.0, 0.05)
    r_bat_range = np.arange(0.0, 1.0, 0.05)

    min_custo = float("inf")
    best_r_pv = 0
    best_r_bat = 0
    for r_pv in r_pv_range:
        for r_bat in r_bat_range:
            custo = calcula_funcao_objectivo(r_pv, r_bat, k, params)
            #print(f"R_pv: {r_pv:.2f}, R_bat: {r_bat:.2f}, Custo Anual: {custo:.2f} €")
            if custo < min_custo:
                min_custo = custo
                best_r_pv = r_pv
                best_r_bat = r_bat

    return best_r_pv, best_r_bat, min_custo

In [23]:
r_pv, r_bat, custo = encontra_indices_minimizam_custo(params)
custo_kwh = custo / params["consumo_anual"]
print(f"Melhor R_pv: {r_pv:.2f}, Melhor R_bat: {r_bat:.2f}, Custo Anual: {custo:.2f} €", f"Custo por kWh: {custo_kwh:.2f} €/kWh")

Melhor R_pv: 1.60, Melhor R_bat: 0.35, Custo Anual: 333.56 € Custo por kWh: 0.11 €/kWh
